In [ ]:
# supervisor

# wedding venue
# DJ - playlist 
# Flights

# tools
# - find wedding venues
# - find DJs
# - find flights

# prompt - You are a wedding planner coordinator. You can call subagents to help you with the planning.


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain.tools import tool 
from typing import Dict, Any 
from tavily import TavilyClient

tavily_client = TavilyClient()

tools = []

@tool
def search(query: str) -> Dict[str, Any]:
    """Searches web for info"""
    return tavily_client.search(query)


In [5]:
from langchain.agents import create_agent

wedding_venue_agent = create_agent(
    model='gpt-5-nano',
    tools=[search]
)

dj_agent = create_agent(
    model='gpt-5-nano',
    tools=[search]
)

flights_agent = create_agent(
    model='gpt-5-nano',
    tools=[search]
)



In [16]:
from langchain.messages import HumanMessage

@tool
def call_wedding_venue_agent(location: str) -> str:
    """Call wedding venue subagent to find the best venue for the wedding."""
    response = wedding_venue_agent.invoke({"messages": [HumanMessage(content=f"Find the best venue for the wedding, Location: {location}")]})
    return response["messages"][-1].content

@tool
def call_dj_agent(specifications: str) -> str:
    """Call dj subagent to ask to find a playlist."""
    response = dj_agent.invoke({"messages": [HumanMessage(content=f"Find a playlist for the wedding. Specifications: {specifications}")]})
    return response["messages"][-1].content

@tool
def call_flights_agent(specifications: str) -> str:
    """Call flights subagent to find the best flights for the wedding."""
    response = flights_agent.invoke({"messages": [HumanMessage(content=f"Find the best flights for the wedding. Specifications: {specifications}")]})
    return response["messages"][-1].content


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
main_agent = create_agent(
    model='gpt-5-nano',
    tools=[
        call_wedding_venue_agent, 
        call_dj_agent, 
        call_flights_agent
    ],
    checkpointer=InMemorySaver(),
    system_prompt="""
        You are a wedding planning agent that can call subagents to delegate tasks -
        find a venue, DJ, and flights for a wedding.
    """
)

In [18]:
config = {"configurable": {"thread_id": "1"}}

response = await main_agent.ainvoke(
    {"messages": [HumanMessage(content="plan a wedding in San Francisco")]},
    config
)

In [9]:
from pprint import pprint 
pprint(response)

{'messages': [HumanMessage(content='plan a wedding in San Francisco', additional_kwargs={}, response_metadata={}, id='c36c4eb7-c555-44d6-81eb-2637017e905b'),
              AIMessage(content='Fantastic! I can plan a great wedding in San Francisco. To tailor it well, I’ll need a few details. In the meantime, here’s a practical, phased plan you can use, plus questions to refine.\n\nWhat I need from you (quick questions)\n- Estimated guest count (range is fine)\n- Target date or date range and how flexible it is\n- Rough budget range for the entire wedding (venue, food/beverage, and extras)\n- Vibe or style you want (classic/elegant, modern/industrial, waterfront/romantic, garden/Presidio, etc.)\n- Preferred neighborhoods or types of venues (historic landmark, hotel ballroom, loft/warehouse, outdoors with tent, etc.)\n- Ceremony preference (same venue as reception or a separate spot)\n- Catering style (sit-down dinner, family-style, or buffet) and whether you want bar options (open bar, li